# Jalachakra Data Integration

This notebook audits the repository's existing data layers and builds only derived outputs under `datasets/curated`. It does not modify raw or cleaned inputs, train models, or construct CWR targets.

## 1. Load cleaned data

Load the cleaned climate, soil, calendar, and growth-stage references. Production crop-location evidence is supplied by the project's wide district-yield source and is reshaped by the integration builder because no cleaned long-form crop-location table exists.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CLEANED_DIR = PROJECT_ROOT / 'datasets' / 'cleaned'
CURATED_DIR = PROJECT_ROOT / 'datasets' / 'curated'
cleaned = {
    'climate': pd.read_csv(CLEANED_DIR / 'Climate_Data_cleaned.csv'),
    'soil': pd.read_csv(CLEANED_DIR / 'Soil_data_cleaned.csv'),
    'crop_calendar': pd.read_csv(CLEANED_DIR / 'Crop_Calender_cleaned.csv'),
    'growth_stage_reference': pd.read_csv(CLEANED_DIR / 'Karnataka_Crop_Growth_Stage_Reference_cleaned.csv'),
}
{name: frame.shape for name, frame in cleaned.items()}

## 2. Normalize crop labels

Aliases are explicit and retained beside canonical names. Unrecognized crop names receive format-only normalization and are not silently treated as an agronomic equivalence.

In [ ]:
from ml.data_integration import _canonical_crop

crop_labels = sorted(set(cleaned['crop_calendar']['Crop'].dropna()) | set(cleaned['growth_stage_reference']['Crop'].dropna()))
crop_normalization_preview = pd.DataFrame([
    {'original_crop_name': label, 'canonical_crop_name': _canonical_crop(label)[0],
     'mapping_reason': _canonical_crop(label)[1], 'mapping_status': _canonical_crop(label)[2]}
    for label in crop_labels
])
crop_normalization_preview

## 3. Normalize location labels

Only capitalization and whitespace are normalized. Historical district names and source codes are kept; no current-boundary crosswalk is inferred.

In [ ]:
crop_location_source = pd.read_csv(PROJECT_ROOT / 'datasets' / 'raw' / 'Crops_data.csv')
location_labels = crop_location_source[['State Name', 'Dist Name', 'State Code', 'Dist Code']].drop_duplicates()
location_normalization_preview = location_labels.rename(columns={
    'State Name': 'original_state', 'Dist Name': 'original_district',
    'State Code': 'state_code', 'Dist Code': 'district_code'
}).copy()
location_normalization_preview['canonical_state'] = location_normalization_preview['original_state'].astype(str).str.strip().str.upper().str.replace(r'\s+', ' ', regex=True)
location_normalization_preview['canonical_district'] = location_normalization_preview['original_district'].astype(str).str.strip().str.upper().str.replace(r'\s+', ' ', regex=True)
location_normalization_preview.head()

## 4. Build crop-location evidence

The wide district-year crop triplets are reshaped with the project's existing source logic. Aggregate crop series and rows without positive reported area are excluded from represented-crop evidence; source row identifiers remain lineage only, never join keys.

In [ ]:
from ml.data_integration import build_curated_layer

tables = build_curated_layer(project_root=PROJECT_ROOT, export=False)
evidence = tables['crop_location_evidence']
evidence.head()

## 5. Build crop reference catalog

Stage boundaries and numeric Kc anchors come from the processed stage reference used by the actual pipeline. Blank anchors remain blank; interpolated Kc values are not fabricated here.

In [ ]:
crop_reference_catalog = tables['crop_reference_catalog']
crop_reference_catalog.groupby(['canonical_crop', 'cwr_support_status'], dropna=False).size().rename('records').reset_index()

## 6. Build soil context

The cleaned soil rows have no verified state, sample date, or field linkage. They are preserved as provisional district context and are not joined to crop evidence.

In [ ]:
soil_context = tables['soil_context_reference']
soil_context[['state_if_verified', 'district', 'nitrogen', 'phosphorus', 'potassium', 'ph', 'data_status']].head()

## 7. Build rainfall context

Daily district observations and annual rainfall normals are separate tables. Weekly, monthly, and cumulative values are not converted into daily records.

In [ ]:
rainfall_context = tables['rainfall_context']
rainfall_normals_context = tables['rainfall_normals_context']
{
    'daily_periods': rainfall_context['period_type'].value_counts().to_dict(),
    'normal_periods': rainfall_normals_context['period_type'].value_counts().to_dict(),
    'daily_sample': rainfall_context.head(3),
    'annual_normal_sample': rainfall_normals_context.head(3),
}

## 8. Build weather context

Cleaned climate values are retained without attaching a site or asserting units. Location, units, and provenance remain explicitly unknown or unverified.

In [ ]:
weather_context = tables['weather_context']
weather_context[['date', 'tavg', 'tmin', 'tmax', 'site_id', 'location_status', 'units_status', 'provenance_status']].head()

## 9. Validate joins

Each candidate join records its key, cardinality, match counts, temporal/spatial/unit compatibility, and decision. Unsafe joins are documented as rejected and are not executed.

In [ ]:
join_validation_report = tables['join_validation_report']
join_validation_report

## 10. Check duplicate expansion

All accepted reference joins are many-to-one. The source crop reshape is intentional wide-to-long conversion; its output is checked against positive-area source triplets.

In [ ]:
safe_joins = join_validation_report[join_validation_report['status'].eq('SAFE')]
assert safe_joins['duplicate_expansion'].eq(0).all()
evidence_key = ['state_code', 'district_code', 'year', 'canonical_crop', 'season']
assert not evidence.duplicated(evidence_key).any()
safe_joins[['join_key', 'cardinality', 'matched_rows', 'duplicate_expansion']]

## 11. Check unmatched records

Unmatched and rejected relations remain visible. Missing references are not filled with guessed crop coefficients or support labels.

In [ ]:
unmatched = join_validation_report.loc[
    join_validation_report['unmatched_rows'].gt(0),
    ['left_dataset', 'right_dataset', 'join_key', 'unmatched_rows', 'status']
]
unmatched

## 12. Export curated data

Only derived files are written to `datasets/curated`. Raw and cleaned inputs are never opened for writing.

In [ ]:
exported_tables = build_curated_layer(project_root=PROJECT_ROOT, output_dir=CURATED_DIR, export=True)
sorted(path.name for path in CURATED_DIR.glob('*'))

## 13. Generate summary statistics

Summarize curated table sizes, crop support categories, and join decisions. Regional crop evidence is not a farm suitability ranking.

In [ ]:
summary = pd.DataFrame([
    {'dataset': name, 'rows': len(frame), 'columns': len(frame.columns)}
    for name, frame in exported_tables.items()
])
print('Curated table sizes')
display(summary)
print('Crop support status')
display(crop_reference_catalog.groupby('cwr_support_status')['canonical_crop'].nunique())
print('Join decisions')
display(join_validation_report['status'].value_counts())